In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def build_cost_base(df_raw, df_flag, rows, d30):

    flag = ('30d' if d30 else '90d')

    cols = ['stay_id','cost_per_day_stay','total_readmission_cost','avg_cost_of_prev_stays'] 
    
    df = df_raw.loc[rows, cols].copy()

    df['readmit_' + flag] = df_flag.loc[rows, 'readmit_' + flag]

    return df

In [11]:
def attach_predictions(df, df_pred, d30 = True):

    flag = ('d30' if d30 else 'd90')

    cols_flag = [col for col in df_pred if flag in col]

    df[cols_flag] = df_pred[cols_flag]

In [12]:
def cost_reduction_preprocessor(df_raw, df_pred, df_flag, d30 = True):

    rows = df_pred.index

    df = build_cost_base(df_raw, df_flag, rows, d30)

    df = attach_predictions(df, df_pred, d30)

    return df

In [13]:
def separate_model_threshold(name):

    underscore_pos = name.rfind("_")
    model = name[:underscore_pos]
    threshold = float(name[(underscore_pos + 1):])

    return model, threshold

In [14]:
def estimate_intervention_cost(row, pred_values, df_cost, model, threshold):

    relative_prob = pred_values.loc[row, model] / threshold

    if(relative_prob < 2 * threshold):

        intervention_cost = min(df_cost.loc[row, 'cost_per_day_stay'], df_cost.loc[row, 'avg_cost_of_prev_stays'])

    elif(relative_prob < 3 * threshold):

        intervention_cost = 2 * min(df_cost.loc[row, 'cost_per_day_stay'], df_cost.loc[row, 'avg_cost_of_prev_stays'])
                    
    else:

        intervention_cost = 3 * min(df_cost.loc[row, 'cost_per_day_stay'], df_cost.loc[row, 'avg_cost_of_prev_stays'])

    return intervention_cost

In [15]:
def estimate_gain(row, col, pred_values, df_cost, gains, model, threshold, r):

    if(df_cost.loc[row, col] == 1):

        exp_avoided_cost = r * pred_values.loc[row, model] * df_cost.loc[row, 'total_readmission_cost']

        intervention_cost = estimate_intervention_cost(row, pred_values, df_cost, model, threshold)

        gains.loc[id, col] = exp_avoided_cost - intervention_cost

    else: 

        gains.loc[id, col] = 0

    return gains

In [16]:
def estimate_cost_reduction(df_cost, pred_values, d30 = True, r = 0.2):

    flag = ('d30' if d30 else 'd90')

    gains = pd.DataFrame(index = df_cost.index)

    for col in df_cost.columns:

        if (flag in col):

            model, threshold = separate_model_threshold(col)

            for row in df_cost.index:

                gains = estimate_gain(row, col, pred_values, df_cost, gains, model, threshold, r)
                
    gains.loc['total_avoided'] = gains.sum(axis = 0)

    return gains

In [ ]:
def estimate_cost_reduction_map(df_cost, df_pred, rmap, d30 = True):

    gains = {}

    for r in rmap:

        gains[r] = estimate_cost_reduction(df_cost, df_pred, d30, r)

    return gains